# Fase 01: Infraestructura y Pipeline de Datos (ETL)

En este notebook consolidamos el proceso de extracción, transformación y carga (incremental) desarrollado en la carpeta `Miguel/ETL`.

## 1. Extracción con Backoff Exponencial

In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import boto3
from dotenv import load_dotenv

# Cargamos variables de entorno para obtener la API KEY
load_dotenv()
API_KEY = os.getenv('AEMET_API_KEY')
headers = {'cache-control': "no-cache", 'api_key': API_KEY, 'accept': "application/json"}

def obtener_valores_climaticos_todas(fecha_ini_utc, fecha_fin_utc, max_retries=15, base_delay=5.0):
    """Descarga datos climáticos de todas las estaciones manejando los límites de la API."""
    url_base = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini_utc}/fechafin/{fecha_fin_utc}/todasestaciones"
    delay = base_delay
    for intento in range(max_retries):
        try:
            response = requests.get(url_base, headers=headers, timeout=30)
            if response.status_code == 200:
                meta = response.json()
                estado = meta.get('estado')
                if estado == 200:
                    url_final = meta.get('datos')
                    datos_response = requests.get(url_final, timeout=30)
                    if datos_response.status_code == 200:
                        return datos_response.json()
                elif estado == 429:
                    print(f"Intento {intento+1}: La API dice Estado 429 (Límite superado).")
            elif response.status_code == 429:
                print(f"Intento {intento+1}: HTTP 429 (Muchas peticiones).")
        except Exception as e:
            print(f"Intento {intento+1}: Error de red/conexión: {e}")
            
        if intento < max_retries - 1:
            print(f"Esperando {delay} segundos antes de reintentar...")
            time.sleep(delay)
            delay *= 2.0
            
    print("No se pudo descargar el bloque.")
    return None

## 2. Transformación y Limpieza de Datos

In [ ]:
def limpiar_y_cargar_datos(datos_json):
    """Limpia el JSON y devuelve un DataFrame formateado."""
    if not datos_json:
        return pd.DataFrame()
        
    df = pd.DataFrame(datos_json)
    
    # Ajustes básicos
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["indicativo"] = df["indicativo"].astype(str)
    
    def a_float(columna):
        if columna is None:
            return np.nan
        return columna.astype(str).str.replace(",", ".").replace("", "nan").astype(np.float32)

    columnas_numericas = ["tmed", "tmin", "tmax", "velmedia", "racha", "sol", "presMax", "presMin", "hrMedia", "hrMax", "hrMin"]
    for col in columnas_numericas:
        if col in df.columns:
            df[col] = a_float(df[col])
            
    if "prec" in df.columns:
        df["prec"] = df["prec"].astype(str).str.replace("Ip", "0.0").str.replace(",", ".")
        df["prec"] = pd.to_numeric(df["prec"], errors="coerce").astype(np.float32)

    return df

## 3. Carga Incremental y Subida a S3

In [ ]:
def integrar_y_guardar_datos(df_lote, anio):
    """Guarda los datos a local y los sube a AWS S3."""
    csv_path = f'climaticos_{anio}.csv'
    
    # Eliminamos duplicados
    df_lote = df_lote.drop_duplicates(subset=["fecha", "indicativo"], keep="last")
    df_lote = df_lote.sort_values(by="fecha", ascending=True)

    print(f"Guardando CSV local para {anio}...")
    df_lote.to_csv(csv_path, index=False)
    
    # Subida a S3 (Descomentar para usar en producción)
    # try:
    #     s3 = boto3.client("s3")
    #     bucket = "carmen-proyecto-aemet-2025"
    #     s3.upload_file(csv_path, bucket, f"csv/climaticos_{anio}.csv")
    #     print(f"Subido a S3 con éxito.")
    # except Exception as e:
    #     print(f"Error subiendo a S3: {e}")